# 3.1 Gate, highway, or route: does a mechanism earn its weights?

At the end of this notebook you can do three things:

- **Point at the control of an architecture claim**: the same model with the new part taken
  out and its weights put back. A model that beats a smaller one may win only because it is
  bigger; the control tells the two apart.
- **Draw a highway and a mixture of experts as routes**, point at the routes that are added
  and at the shapes that must match.
- **Tell the weights a sparse mixture of experts trains from the weights it uses**, say why
  using fewer weights does not make a model faster by itself, recognise router collapse and
  say what load balancing does about it.

The exercise of this lesson (`05_exercise`) asks you to put a routed architecture against a
control on the heartbeats. This notebook does the same on a small forecasting problem.

**How this notebook runs.** The models and their training live in `gates.py`. Two scripts
train them, `gates_family.py` and `gates_balance.py`, and store their results in `results/`;
this notebook draws every figure and table from those files. `RERUN = True` in the next cell
trains everything again: about 8 minutes for the family and 10 for balancing.

In [ ]:
import subprocess
import sys
from pathlib import Path

from notebooktester import param

# True: train every model again with both scripts (about 8 minutes for the family, 10 for
# balancing) before loading their results. Under notebooktester both scripts run in a test mode of seconds and store their
# results apart, in results/test.
RERUN = param(False, test=True)
RESULTS = Path("results", param("", test="test"))
if RERUN:
    for script in ("gates_family.py", "gates_balance.py"):
        subprocess.run([sys.executable, script], check=True)

## 3.1.0 The data and the task

The series is the electricity demand of one Australian state (series `T1` of the Monash
forecasting archive), in megawatts (MW), every half hour from 2002 to early 2015. Demand
follows people: low at night, high in the day, lower at weekends.

**The task.** From four days of demand (192 half-hours) predict the next day (48). One
example is a **window** of 240 consecutive half-hours: 192 input, 48 target. A model maps a
`(B, 192)` batch to a `(B, 48)` batch.

**The split in time.** The last full year is **test**, the year before it **val**,
everything earlier **train**: a forecast is used on days after the data it learned from. As
in lesson 2, train fits the weights, val chooses (the epoch to keep, the better model), test
is scored once at the end.

**Windows and scale.** Train windows start every 24 half-hours, val and test windows one per
day. Values are scaled with the mean and standard deviation of the train span. The models
train on the mean squared error (MSE) and are reported in the mean absolute error (MAE), the
average miss.

The next cell loads the series, cuts the windows (`gates.load_windows`) and draws both.

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from gates import DAY, H, W, load_windows

data = load_windows()
print(data.series)
print(
    f"train mean {data.mu:.0f} MW, train std {data.sigma:.0f} MW: "
    f"an MAE of 0.1 is {0.1 * data.sigma:.0f} MW"
)
print(
    f"train {tuple(data.xtr.shape)}   val {tuple(data.xva.shape)}   "
    f"test {tuple(data.xte.shape)}"
)

series = data.series
fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 6))
daily = series.values.reshape(-1, DAY).mean(1).numpy()
days = series.timestamps[::DAY]
for split, colour in [("train", "C0"), ("val", "C1"), ("test", "C2")]:
    s = series.splits[split]
    part = slice(s.start // DAY, s.stop // DAY)
    top.plot(days[part], daily[part], color=colour, lw=0.6, label=split)
top.set_ylabel("daily mean demand (MW)")
top.set_title(
    "The whole series, one point per day: train, then a year of val, then a year of test"
)
top.legend(fontsize=8, loc="upper left")

first = 10  # one val window, drawn in MW
hours = np.arange(W + H) / 2
xs = data.xva[first] * data.sigma + data.mu
ys = data.yva[first] * data.sigma + data.mu
bottom.plot(hours[:W], xs, color="C1", label="input: 192 half-hours (4 days)")
bottom.plot(hours[W:], ys, color="k", label="target: the next 48 half-hours (1 day)")
bottom.axvline(W / 2, color="grey", ls="--", lw=1)
bottom.set_xlabel("hours from the start of the window")
bottom.set_ylabel("demand (MW)")
bottom.set_title("One val window: the model sees four days and predicts the fifth")
bottom.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

The top panel shows demand rising and falling over the year; val and test are the last two
years. The bottom panel shows a daily cycle in each input day, and the target day continues
it. A model that has learned the daily shape and the recent level already does much of the
job. The question of this notebook is whether a gate or a router adds anything to that.

## 3.1.1 One gate, three uses: gate, highway, mixture of experts

Lesson 2 (2.2.3) built a **gate**: `y = σ(W x + b) ⊙ x`. A `Linear` computes one number per
element, the sigmoid turns each into a multiplier between 0 and 1, and each element is
multiplied by its own. Put the same multiplier between two routes that are added, and it
becomes part of a **parallel architecture**:

<svg width="720" height="520" viewBox="0 0 720 520" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="720" height="520" fill="white"/>
<defs><marker id="m1" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="10" y="20" font-weight="bold" fill="#222">highway (vectormesh Highway)</text>
<text x="10" y="104" fill="#222">x</text>
<text x="4" y="124" font-family="monospace" font-size="10" fill="#222">(B, 64)</text>
<line x1="22" y1="100" x2="48" y2="100" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="50" y="88" width="70" height="24" rx="6" fill="#e4e4e4" stroke="#555"/>
<text x="85" y="104" text-anchor="middle" font-size="11" fill="#222">LayerNorm</text>
<line x1="120" y1="100" x2="135" y2="100" stroke="#444" stroke-width="1.5"/>
<path d="M 135 100 L 135 55 L 168 55" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="170" y="35" width="130" height="40" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="52" text-anchor="middle" fill="#222">block T</text>
<text x="235" y="67" text-anchor="middle" font-size="10" fill="#222">Linear, GELU, Linear</text>
<line x1="300" y1="55" x2="361" y2="55" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="331" y="47" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<circle cx="380" cy="55" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="60" text-anchor="middle" fill="#222">× g</text>
<line x1="135" y1="100" x2="168" y2="100" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="170" y="87" width="130" height="26" rx="8" fill="#f3d9e6" stroke="#555"/>
<text x="235" y="104" text-anchor="middle" font-size="11" fill="#222">g = σ(Linear(64, 64))</text>
<line x1="300" y1="100" x2="368" y2="74" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<line x1="300" y1="100" x2="366" y2="131" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<text x="235" y="127" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">g: (B, 64)</text>
<path d="M 135 100 L 135 150 L 359 150" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="150" y="145" font-size="11" fill="#222">skip: LN(x) itself, (B, 64), no weights</text>
<circle cx="380" cy="150" r="20" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="154" text-anchor="middle" font-size="11" fill="#222">× (1−g)</text>
<path d="M 398 55 L 470 55 L 470 82" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<path d="M 400 150 L 470 150 L 470 118" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<circle cx="470" cy="100" r="16" fill="#dccdf0" stroke="#555"/>
<text x="470" y="106" text-anchor="middle" font-size="18" fill="#222">+</text>
<line x1="486" y1="100" x2="588" y2="100" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="537" y="92" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<text x="596" y="104" fill="#222">y</text>
<text x="10" y="186" font-size="12" fill="#222">y = g ⊙ T(LN(x)) + (1 − g) ⊙ LN(x): one weight per element; the two weights of an element sum to 1</text>
<line x1="10" y1="203" x2="710" y2="203" stroke="#bbb" stroke-width="1"/>
<text x="10" y="224" font-weight="bold" fill="#222">mixture of experts (MoE)</text>
<text x="10" y="304" fill="#222">x</text>
<line x1="24" y1="300" x2="90" y2="300" stroke="#444" stroke-width="1.5"/>
<text x="28" y="292" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<line x1="90" y1="250" x2="90" y2="400" stroke="#444" stroke-width="1.5"/>
<line x1="90" y1="250" x2="168" y2="250" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="90" y1="300" x2="168" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="90" y1="350" x2="168" y2="350" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="90" y1="400" x2="168" y2="400" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="170" y="236" width="130" height="28" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="254" text-anchor="middle" font-size="12" fill="#222">expert 1: Linear(64, 64)</text>
<rect x="170" y="286" width="130" height="28" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="304" text-anchor="middle" font-size="12" fill="#222">expert 2: Linear(64, 64)</text>
<text x="235" y="331" text-anchor="middle" fill="#222">⋮</text>
<rect x="170" y="336" width="130" height="28" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="354" text-anchor="middle" font-size="12" fill="#222">expert N: Linear(64, 64)</text>
<line x1="300" y1="250" x2="361" y2="250" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="300" y1="300" x2="361" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="300" y1="350" x2="361" y2="350" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="331" y="242" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<circle cx="380" cy="250" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="255" text-anchor="middle" fill="#222">× g₁</text>
<circle cx="380" cy="300" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="305" text-anchor="middle" fill="#222">× g₂</text>
<circle cx="380" cy="350" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="354" text-anchor="middle" font-size="11" fill="#222">× g_N</text>
<rect x="170" y="386" width="150" height="28" rx="8" fill="#f3d9e6" stroke="#555"/>
<text x="245" y="404" text-anchor="middle" font-size="11" fill="#222">router: Linear(64, N), softmax</text>
<line x1="320" y1="400" x2="340" y2="400" stroke="#444" stroke-width="1.5"/>
<circle cx="340" cy="400" r="3" fill="#444"/>
<line x1="340" y1="400" x2="340" y2="262" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3"/>
<line x1="340" y1="262" x2="364" y2="259" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<line x1="340" y1="312" x2="364" y2="309" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<line x1="340" y1="362" x2="364" y2="359" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<text x="398" y="278" font-family="monospace" font-size="10" fill="#222">g₁ (B, 1)</text>
<text x="398" y="328" font-family="monospace" font-size="10" fill="#222">g₂ (B, 1)</text>
<text x="398" y="378" font-family="monospace" font-size="10" fill="#222">g_N (B, 1)</text>
<text x="352" y="420" font-family="monospace" font-size="10" fill="#222">g (B, N), split into N columns; the N sum to 1</text>
<path d="M 398 250 L 470 250 L 470 282" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="398" y1="300" x2="452" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<path d="M 398 350 L 470 350 L 470 318" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<circle cx="470" cy="300" r="16" fill="#dccdf0" stroke="#555"/>
<text x="470" y="306" text-anchor="middle" font-size="18" fill="#222">+</text>
<line x1="486" y1="300" x2="588" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="537" y="292" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<text x="596" y="304" fill="#222">y</text>
<text x="10" y="452" font-size="12" fill="#222">y = g₁ · expert₁(x) + … + g_N · expert_N(x), with g = softmax(router(x)): one weight per expert, per window;</text>
<text x="10" y="470" font-size="12" fill="#222">each g_i, (B, 1), multiplies all 64 features of its expert's (B, 64) output (broadcast)</text>
<text x="10" y="496" font-size="12" fill="#222">sparse (k = 1): per window only the expert with the largest g_i runs, its output times g_i; the other routes are not computed</text>
</svg>

**Highway** (lesson 2, 2.2.3, wrote the gate `Γ`; here it is `g`). Two routes leave `x`:
through a block `T`, and around it, the skip. The gate gives one multiplier per element,
`g = σ(Linear(64, 64)(x))`, and `y = g ⊙ T(x) + (1 − g) ⊙ x`: where `g` is near 1 the block
passes, near 0 the input passes unchanged. vectormesh's `Highway` puts a LayerNorm on `x`
first, as its `Skip` does (2.2.1.3).

**From a highway to a mixture of experts** (MoE; Jacobs and colleagues, 1991). Change three
things:

- **The routes become experts**: `N` blocks of the same kind, here each a `Linear(64, 64)`.
- **The skip has no counterpart**: every route of a MoE is a block with weights.
- **The gate becomes a router**, a `Linear(64, N)` with a softmax. Where the highway gives
  every element its own `g` and `1 − g`, the router gives every window one weight per route,
  `g_1 … g_N`, summing to 1 over the routes. Each `g_i` is `(B, 1)` and multiplies all 64
  features of its expert's output (broadcasting).

A **dense** MoE (`MoE`) runs every expert on every window. A **sparse** MoE (`SparseMoE`)
runs only the `k` experts with the largest weights; with `k = 1`, as here, one expert runs
and its output is multiplied by its `g_i`.

**What must match.** Routes are added element by element, so every route gives the same
shape, `(B, 64)`; the router gives one weight per expert, `(B, N)`.

### The models in this notebook

Every model is the same frame with something different in the middle:

`(B, 192)` → `Linear(192, 64)` → GELU → **middle** → `Linear(64, 48)` → `(B, 48)`

The first `Linear` turns four days into 64 features, the last turns 64 features into the
48 half-hours of the forecast. The middle takes `(B, 64)` and gives `(B, 64)`:

| model | middle | it has, besides the frame |
|---|---|---|
| `plain` | nothing | no middle at all: the frame alone |
| `deeper` | `Linear(64, 64)`, GELU | one more plain layer |
| `gate` | `Gate(64)`: `σ(Linear(64, 64)(x)) ⊙ x` | the gate's `Linear(64, 64)` |
| `highway` | `Highway`: LayerNorm, gate `Linear(64, 64)`, block `T` = `Linear(64, 64)`, GELU, `Linear(64, 64)` | the gate, the block and the LayerNorm |
| `moe` | dense `MoE`, 4 experts `Linear(64, 64)`, router `Linear(64, 4)` | 4 experts and a router |
| `sparse_moe` | `SparseMoE`, `N` experts `Linear(64, 64)`, router `Linear(64, N)`, `k = 1` | `N` experts and a router |

`deeper (hidden 86)` and `deeper (hidden 96)` are `deeper` with 86 or 96 features in every
layer: the controls of the larger models (3.1.2). The next cell passes eight val windows to the
middle of each of the five models that have one (`gates.make_model`) and prints the shapes.

In [ ]:
from gates import CONTROL, MODEL_CONFIGS, make_model, n_params

windows = data.xva[:8]
for label, mechanism, kwargs, _ in MODEL_CONFIGS:
    if mechanism not in ("gate", "highway", "moe", "sparse_moe"):
        continue
    inp, act, middle, _ = make_model(mechanism, **kwargs)
    x = act(inp(windows))
    line = f"{label:18s} middle {tuple(x.shape)} -> {tuple(middle(x).shape)}"
    if "moe" in mechanism:
        line += (
            f"   an expert {tuple(middle.experts[0](x).shape)}"
            f"   router {tuple(middle.router(x).shape)}"
        )
    print(line)

## 3.1.2 The family, each model with its control

A model with more weights can fit more, so a model with a new part that beats a smaller one
may win only because it is larger. The **control** takes the part out and puts its weights
back, as a plain layer or as width. It differs from its arm in the part, not in the number of
weights:

- `gate` against `deeper`: the gate's `Linear(64, 64)` comes back as a plain layer;
- `highway` against `deeper (hidden 86)`, `moe` and `sparse_moe (N=4)` against
  `deeper (hidden 96)`: the weights come back as width;
- `sparse_moe (N=64)` has no control; 3.1.6 counts how many of its weights one window uses.

`plain`, the frame alone, is the **baseline** every model must beat. "Weights" means every
number the optimizer changes: a `Linear(64, 64)` has 64 × 64 + 64 = 4,160. The next cell
counts them.

In [ ]:
PARAMS = {
    label: n_params(make_model(mechanism, **kwargs))
    for label, mechanism, kwargs, _ in MODEL_CONFIGS
}

print(
    f"{'model':20s} {'frame':>7s} {'middle':>8s} {'total':>8s}   control (total, difference)"
)
for label, mechanism, kwargs, control in MODEL_CONFIGS:
    h = kwargs.get("hidden", 64)
    frame = (W * h + h) + (h * H + H)
    against = (
        f"{control} ({PARAMS[control]:,}, {PARAMS[control] - PARAMS[label]:+,})"
        if control
        else ""
    )
    print(
        f"{label:20s} {frame:7,} {PARAMS[label] - frame:8,} {PARAMS[label]:8,}   {against}"
    )

The controls match: `gate` and `deeper` both have 19,632 weights; `highway` has 176 fewer than
`deeper (hidden 86)`, and `moe` and `sparse_moe (N=4)` 124 fewer than `deeper (hidden 96)`. A
wider control also differs from its arm in more than the part: its weights sit partly in the
frame, `highway` has a LayerNorm and the experts have no GELU. Everything else is held equal:
the windows, the seeds, Adam with a learning rate of 1e-3, batches of 64, the MSE loss, and the
training (3.1.3).

## 3.1.3 Train for the whole budget, keep the best epoch

Every run trains for the whole budget of 250 epochs and keeps the weights of the epoch with
the lowest val loss; no run stops early, because a baseline stopped before it has settled
moves every comparison made with it. `gates_family.py` trained each model with seeds 0, 1
and 2 and stored every val curve.

**Before you run the next cells**, write down a ranking of the nine models on val, and for
each of `gate`, `highway`, `moe` and `sparse_moe (N=4)` whether you expect it to beat its
control.

In [ ]:
family = json.loads((RESULTS / "family.json").read_text())
models = family["models"]
SEEDS = family["seeds"]
print(
    f"{len(models)} models x {len(SEEDS)} seeds x {family['epochs']} epochs, "
    f"trained in {family['minutes']} min on {family['threads']} threads"
)

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True, sharey=True)
low = min(r["val_mae"] for m in models.values() for r in m["runs"])
for ax, (label, m) in zip(axes.flat, models.items()):
    for i, r in enumerate(m["runs"]):
        curve = r["val_mae_curve"]
        ax.plot(
            range(1, len(curve) + 1),
            curve,
            color=f"C{i}",
            lw=0.8,
            label=f"seed {r['seed']}",
        )
        ax.plot(r["best_epoch"], r["val_mae"], "o", color=f"C{i}", ms=5)
    ax.set_title(label, fontsize=10)
    ax.set_ylim(low - 0.005, low + 0.065)
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("epoch")
for ax in axes[:, 0]:
    ax.set_ylabel("val MAE")
axes[0, 0].legend(fontsize=7)
fig.suptitle(
    "Val MAE of every run over the whole budget; ● the epoch kept (lowest val loss)"
)
plt.tight_layout()
plt.show()

Every curve falls steeply over the first 50 epochs and bounces from epoch to epoch. `plain` is
still falling at epoch 250 (two of its runs keep epoch 240 or later), so a longer run could
only lower it and make it harder to beat. `deeper (hidden 86)`, `deeper (hidden 96)`, `gate`
and `highway` keep an epoch between 48 and 83 and then climb slowly.

## 3.1.4 The control at work

**Question.** Does any part beat its control, and does any model beat `plain`?

**Setup.** The figure puts the three runs of every arm next to the three runs of its control,
each run a dot at the val MAE of its kept epoch, with `plain` on the left as the baseline.
Lesson 2's rule reads it: with three seeds and a spread of about 0.002 per model, the standard
error of a difference is about 0.0016, so a difference counts beyond about 0.003.

In [ ]:
OKABE_ITO = {"plain": "#000000", "arm": "#E69F00", "control": "#0072B2"}
PAIRS = [
    ("plain", None),
    ("gate", "deeper"),
    ("highway", "deeper (hidden 86)"),
    ("moe", "deeper (hidden 96)"),
    ("sparse_moe (N=4)", "deeper (hidden 96)"),
    ("sparse_moe (N=64)", None),
]
val = {
    label: np.array([r["val_mae"] for r in m["runs"]]) for label, m in models.items()
}

fig, ax = plt.subplots(figsize=(12, 4.5))
ticks, names = [], []
for i, (arm, control) in enumerate(PAIRS):
    columns = [(arm, "plain" if arm == "plain" else "arm", 0.0)]
    if control:
        columns.append((control, "control", 0.4))
    for label, role, dx in columns:
        x = i + dx
        v = val[label]
        ax.scatter(np.full(len(v), x), v, color=OKABE_ITO[role], s=30, zorder=3)
        ax.hlines(v.mean(), x - 0.12, x + 0.12, color=OKABE_ITO[role], lw=2)
        ticks.append(x)
        names.append(label)
ax.axhline(val["plain"].mean(), color="grey", ls="--", lw=1, label="mean of plain")
for role, colour in OKABE_ITO.items():
    ax.scatter([], [], color=colour, label=role)
ax.set_xticks(ticks, names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("val MAE at the kept epoch")
ax.set_title("Every run as a dot, the mean as a bar: each arm next to its control")
ax.legend(fontsize=8, ncol=4, loc="upper left")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(
    f"{'model':20s} {'weights':>9s} {'val MAE':>16s}   control, its val MAE, arm minus control"
)
for label, *_ in MODEL_CONFIGS:
    v = val[label]
    control = CONTROL[label]
    against = (
        f"{control}, {val[control].mean():.4f}, {v.mean() - val[control].mean():+.4f}"
        if control
        else ""
    )
    print(
        f"{label:20s} {PARAMS[label]:9,} {v.mean():.4f} ± {v.std(ddof=1):.4f}   {against}"
    )

- `gate` (0.1508) is level with `plain` (0.1510) and 0.0022 above `deeper`, its control. As a
  plain layer the same 4,160 weights put `deeper` 0.0023 below `plain`, within the spread:
  neither buys anything measurable.
- `highway` sits 0.0021 below `plain`, which read alone looks like a gain. Its control
  `deeper (hidden 86)` sits 0.0024 below `plain`: whatever is there belongs to the weights, not
  the highway, and neither gap is beyond the spread.
- `moe` sits 0.0021 below `plain` and 0.0016 below its control `deeper (hidden 96)`, both within
  the spread.
- Both sparse models lie highest (0.1558 and 0.1556); `sparse_moe (N=4)` is 0.0052 above its
  control, beyond the spread. 3.1.5 shows what their routers do.

### The test year, once

The test year took part in no choice: not in the epoch kept, not in any reading above. Here it
is scored once, with the weights each run kept.

In [ ]:
for label, m in models.items():
    t = np.array([r["test_mae"] for r in m["runs"]])
    print(f"{label:20s} val MAE {val[label].mean():.4f}   test MAE {t.mean():.4f}")

Every model has a higher MAE on test (0.1555 to 0.1635) than on val (0.1486 to 0.1558), partly
because the epoch was chosen on val and partly because the years differ. The sparse models stay
the worst, while `highway` and `moe` change sides with their controls, as gaps within the spread
can.

## 3.1.5 Router collapse and load balancing

A sparse router sends each window to one expert. If it sends all windows to the same one,
the model has turned into a single expert and the others are weights that never see data:
**router collapse**. It could come from the random start or from training.

The next cell counts, for each seed's `sparse_moe` with 4 and with 64 experts, how many of the
361 val windows its router sends to each expert, untrained and trained. It also prints the
mean **top weight**: the router's weight `g_i` of the chosen expert, the number its output is
multiplied by.

**Predict first**: are more or fewer experts in use after training than before?

In [ ]:
def describe(counts: list[int], top_weight: float) -> str:
    used = sum(c > 0 for c in counts)
    return (
        f"{used:2d} of {len(counts)} experts used   "
        f"largest expert {max(counts):3d} of {sum(counts)} windows   "
        f"top weight {top_weight:.3f}"
    )


for label in ("sparse_moe (N=4)", "sparse_moe (N=64)"):
    print(label)
    for route in models[label]["routing"]:
        before, after = route["untrained"], route["trained"]
        print(
            f"  seed {route['seed']}  untrained: {describe(before['counts'], before['top_weight'])}"
        )
        print(f"          trained:   {describe(after['counts'], after['top_weight'])}")

**The untrained routers are already uneven, and training collapses them.** Untrained, the
router of `N=4` uses 3 or 4 of its 4 experts, its largest taking 151 to 272 of the 361 windows,
and the one of `N=64` uses 9 to 16 of 64: the random start already favours some experts. After
training, every seed of `N=4` sends all 361 windows to one expert, and every seed of `N=64` 360
or 361. The top weight says the same: untrained about `1/N` (0.278 to 0.293 with 4 experts,
0.023 to 0.024 with 64), trained 0.792 to 0.951, a router nearly certain of its one expert.

So after training `sparse_moe` is the frame with one `Linear(64, 64)` in the middle, its output
multiplied by the top weight; with 64 experts the other 63 are weights no val window reaches.
`moe` and `sparse_moe (N=4)` have the same weights and differ only in routing each window to
one expert, and the second is clearly worse on val, by 0.0069. The usual account of the
collapse is a feedback loop: an expert that receives a few more windows gets more gradient,
becomes better on them, and is chosen more often.

### What a balancing loss does

**How the router learns.** Which expert is largest has no gradient: a small change of the
router's weights does not change the choice. The weight `g_i` that multiplies the chosen
expert's output does have one, so the router learns to give a larger weight to an expert
whose output lowers the loss, and that expert becomes the choice for more windows.

The Switch Transformer (2021) adds an auxiliary loss to the training loss,
`load_balancing_loss`: `N · Σ f_i · P_i`, with `f_i` the fraction of windows sent to expert `i`
and `P_i` the router's mean weight for it. When every expert gets `1/N` of the windows and of
the weight, it is `N · N · (1/N · 1/N) = 1`; when one expert gets everything, it is `N · 1 · 1
= N`. Only `P_i` has a gradient (`f_i` is a count). That gradient is `N · f_i`, so the busiest
expert's mean weight is pushed down hardest, towards `1/N`. It is added with a factor, here
1e-2 as in that paper, and only while training.

**Setup.** `sparse_moe (N=64)`, `k = 1`, balancing weight 1e-2, three seeds
(`gates_balance.py`), against the unbalanced runs of 3.1.3. Experts used, top weight and MAE
are all measured on the val windows.

**Predict first**: with a loss that pushes the router towards many experts, what happens to
the number of experts used, and to the val MAE?

In [ ]:
balance = json.loads((RESULTS / "balance.json").read_text())
without = models["sparse_moe (N=64)"]
arms = {
    "k=1, no balance": [
        (r["val_mae"], route["trained"]["counts"], route["trained"]["top_weight"])
        for r, route in zip(without["runs"], without["routing"])
    ],
    f"k=1, balance {balance['balance']:g}": [
        (r["val_mae"], r["counts"], r["top_weight"]) for r in balance["runs"]
    ],
}
print(
    f"{'sparse_moe (N=64), on val':26s} {'val MAE':>16s} {'top weight':>10s}   experts used per seed"
)
for label, rows in arms.items():
    v = np.array([score for score, _, _ in rows])
    used = [sum(c > 0 for c in counts) for _, counts, _ in rows]
    top = np.mean([w for _, _, w in rows])
    print(f"{label:26s} {v.mean():.4f} ± {v.std(ddof=1):.4f} {top:10.3f}   {used}")

With balancing the windows spread over 22 to 33 experts instead of 1 or 2, and the val MAE
rises from 0.1556 to 0.1719, far beyond the spread. The top weight falls from 0.834 to 0.058:
that is the second change. The chosen expert's output now reaches the last `Linear` multiplied
by about 0.06. This run cannot tell which of the two changes, the spread or the shrinking top
weight, cost the accuracy; E6 separates them.

**Usage is a diagnosis, not a goal.** The count of experts used says whether the router
collapsed; a router that uses more experts is not therefore a better forecaster.

## 3.1.6 Weights trained, weights used, and latency

A sparse MoE trains every expert, but runs only the chosen ones on a window. So it has two
counts: the **total** weights, all that are trained and stored, and the **active** (used)
weights per window, the ones that take part in one window's forecast: the frame, the router
and the `k` chosen experts. A dense MoE runs every expert, so for it the two are the same.

In [ ]:
def active_weights(label: str, n_experts: int, k: int) -> int:
    # Weights one window passes through: the total minus the experts it is not routed to.
    per_expert = 64 * 64 + 64  # Linear(64, 64)
    return PARAMS[label] - (n_experts - k) * per_expert


for label, n_experts, k in [
    ("moe", 4, 4),
    ("sparse_moe (N=4)", 4, 1),
    ("sparse_moe (N=64)", 64, 1),
]:
    active = active_weights(label, n_experts, k)
    print(
        f"{label:18s} total {PARAMS[label]:8,}   active per window {active:7,}   "
        f"total / active {PARAMS[label] / active:5.1f}"
    )

`sparse_moe (N=64)` stores 285,872 weights and uses 23,792 for one window: the frame, the
router `Linear(64, N)` and one expert, 12 times fewer than it has. The router grows with
`N`, so the active count does too, even at one expert per window.

### Does a model that uses fewer weights run faster?

**Latency** is the time of one forward pass. Dense `MoE` runs every expert on the whole batch
and adds the weighted outputs. `SparseMoE`, for every expert that received a window, takes out
those windows (a *gather*), runs the expert on them and adds the result back into place (a
*scatter*).

A CPU runs PyTorch operations on several **threads** (cores working on one operation at
once). For a tiny operation, starting the threads can cost more than the work, so the timing
runs at the default number of threads and at one.

**Setup.** Dense against sparse (`k = 1`, a fresh router), a seeded batch of 64 windows, `N`
from 4 to 256, without gradients; each time is the median of several repeats. This runs live,
in seconds.

**Predict first**: at `N = 4` the sparse middle uses one expert's weights where dense uses
four. Is it faster?

In [ ]:
from functools import partial

from torch import nn
from vectormesh.components import MoE, SparseMoE

N_EXPERTS_TO_CHECK = param((4, 16, 64, 256), test=(4, 16))
CALLS = param(30, test=3)  # forward passes per timing
REPEATS = param(7, test=2)  # timings per arm; the median is reported
DEFAULT_THREADS = torch.get_num_threads()
THREADS = (DEFAULT_THREADS, 1)


def time_call(f) -> float:
    # Median over REPEATS of the mean time of CALLS calls of f(), in ms.
    times = []
    with torch.no_grad():
        for _ in range(5):
            f()
        for _ in range(REPEATS):
            start = time.perf_counter()
            for _ in range(CALLS):
                f()
            times.append((time.perf_counter() - start) / CALLS * 1000)
    return float(np.median(times))


def experts(n: int) -> list[nn.Module]:
    return [nn.Linear(64, 64) for _ in range(n)]


# a seeded batch of this notebook's windows, projected to 64 features once: the block
# under test is the mixture of experts, not the input projection
torch.manual_seed(0)
with torch.no_grad():
    probe_x = nn.Linear(W, 64)(data.xtr[:64])

timings = {t: {"dense": {}, "sparse": {}} for t in THREADS}
for threads in THREADS:
    torch.set_num_threads(threads)
    print(f"{threads} thread(s)")
    for n in N_EXPERTS_TO_CHECK:
        torch.manual_seed(0)
        dense = MoE(experts(n), hidden_size=64, out_size=64)
        torch.manual_seed(0)
        sparse = SparseMoE(experts(n), hidden_size=64, out_size=64, k=1)
        t = timings[threads]
        t["dense"][n] = time_call(partial(dense, probe_x))
        t["sparse"][n] = time_call(partial(sparse, probe_x))
        print(
            f"  N={n:4d}  dense {t['dense'][n]:6.3f} ms   sparse {t['sparse'][n]:6.3f} ms   "
            f"sparse / dense {t['sparse'][n] / t['dense'][n]:4.1f}x"
        )
torch.set_num_threads(DEFAULT_THREADS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, threads in zip(axes, THREADS):
    for name, marker in [("dense", "o"), ("sparse", "s")]:
        ns = list(timings[threads][name])
        ax.plot(ns, [timings[threads][name][n] for n in ns], marker=marker, label=name)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("number of experts N (log)")
    ax.set_title(f"{threads} thread(s): one forward pass of a batch of 64")
    ax.grid(alpha=0.3, which="both")
axes[0].set_ylabel("time per forward pass (ms, log)")
axes[0].legend()
plt.tight_layout()
plt.show()

- **At `N = 4` sparse uses fewer weights and is slower at both thread counts**: 0.763 ms
  against 0.080 ms for dense at 6 threads, 0.086 against 0.058 at 1.
- **At 6 threads** (left, this machine's default) sparse is slower than dense at every `N`,
  from 9.5 times at `N = 4` to 1.8 times at `N = 256`: it starts the threads for many small
  gather and scatter calls.
- **At 1 thread** (right) the gathers and scatters themselves turn out cheap: sparse catches up
  as `N` grows and is faster from `N = 64` on (0.5 and 0.3 times dense), where dense runs 64 or
  256 experts on every window.

Absolute times depend on the machine and on what else runs on it; the shape is the point.
Fewer active weights is not lower latency by itself: measure on the machine and with the
settings the model will run with.

## What you now know

- **The control decides an architecture claim.** It takes the new part out and puts its
  weights back, as a plain layer or as width. Here `highway` sat 0.0021 below `plain`
  and its control `deeper (hidden 86)` 0.0024 below: whatever is there belongs to the weights,
  and neither gap is beyond the spread.
- **A gate is part of a parallel architecture.** A highway weighs a block with `g` and the
  skip with `1 − g`. A mixture of experts turns the routes into experts, has no skip, and
  gives one router weight per expert and window, summing to 1. Every route gives the same
  shape; the router gives `(B, N)`.
- **Total and active weights differ.** `sparse_moe (N=64)` stores 285,872 weights and uses
  23,792 for one window.
- **Router collapse.** After training, every router sent all, or all but one, of the 361 val
  windows to one expert.
- **Balancing** pushes the router's mean weights towards `1/N`. Here it spread the windows,
  lowered the top weight and cost accuracy. Usage is a diagnosis, not a goal.
- **Fewer active weights is not automatically lower latency.** At `N = 4` sparse used fewer
  weights and was slower at both thread counts.

`02_scaling_curves` asks the general version: how the score moves with more parameters, more
data and more compute.

## Exercises

**E1: predict, then measure.** Write down whether you expect `gate` to beat `deeper`. Set
`SEEDS` in `gates_family.py` to ten seeds, rerun it: does any reading of 3.1.4 change?

**E2: force the gate open.** Set the bias of the `Gate`'s `Linear` to 10, so the sigmoid gives
about 1 everywhere, and freeze that layer (`requires_grad_(False)`). The model then computes
what `plain` computes, with 4,160 idle weights. Check after training that the smallest gate
value on val is still about 1. Does its val MAE come out level with `plain`'s?

**E3: more experts, dense.** Train `make_model("moe")` with 2, 4, 8 and 16 experts
(`gates.run`), each against a `deeper` control of the same total. Does the mixture beat its
control at any `N`, or does its score follow its number of weights?

**E4: another state.** Set `australian_electricity.ITEM_ID = "T3"` (of `T1` to `T5`) before
`load_windows()` and train the family with `gates.run`. Does any part beat its control there?

**E5: early stopping on the stored curves.** With `gates.early_stop_epochs(curve, patience)`,
replay patience 10 and 25 on the val-loss curves in `results/family.json`. Which epoch would
each keep, at what val MAE against the whole budget? Would stopping `plain` early have changed
a reading of 3.1.4?

**E6: the dose of balancing, and `k = 2`.** Run `gates_balance.py --balance 1e-4` and
`--balance 1e-3`: how do experts used, top weight and val MAE move with the dose? Then run
`gates_balance.py --k 2 --normalize` with `--balance 0`, `3e-4` and `1e-2`. `normalize`
rescales the two chosen weights to sum to 1, so balancing cannot shrink the signal. Does it
still cost accuracy? (At `k = 2` a window counts for both its experts.)

**E7: threads, an even router and `k = 2`.** Give the timed `SparseMoE` a router that sends
window `i` to expert `i mod N` (a large one-hot, whatever the input), the spread balancing aims
for. Is an even load faster or slower than the router's own? Then time at 1, 2 and 4 threads:
where does sparse start to win? With `k = 2`, does sparse come closer to dense, or move further
away?

**E8: experts with a GELU.** Make every expert `NeuralNet(64, 64)` (a `Linear`, a GELU, a
`Linear`), like `deeper`'s extra layer, with a control of as many weights. Does the gap to the
control change?